# 07 — Final Evaluation

## Objetivo
Evaluar los modelos candidatos de predicción del precio por noche de Airbnb en Buenos Aires, comparar su rendimiento y analizar los errores.

**Nota metodológica:** el conjunto de test se consultó durante experimentos anteriores. Por eso, los resultados de este notebook sirven como evaluación práctica del flujo actual, pero no constituyen una estimación completamente independiente.

## 1. Carga de datos y definición del objetivo

Verificá la ruta y el nombre de la variable objetivo. Este ejemplo asume que el dataset procesado está en `../data/processed/airbnb_dp.csv` y que la columna objetivo se llama `price`. La división debe coincidir con la utilizada en los notebooks anteriores.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error,
    median_absolute_error
)

df = pd.read_csv("../data/processed/airbnb_dp.csv")

TARGET = "price"
if TARGET not in df.columns:
    raise ValueError(
        f"No se encontró la columna objetivo {TARGET!r}. "
        f"Columnas disponibles: {df.columns.tolist()}"
    )

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")


## 2. Modelos candidatos

Se comparan dos versiones de HistGradientBoosting: una con todas las variables y otra con selección automática mediante ExtraTrees. El selector se ajusta solamente con los datos de entrenamiento.

In [ ]:
hgb_params = {
    "l2_regularization": 7.87,
    "learning_rate": 0.068,
    "max_iter": 140,
    "max_leaf_nodes": 38,
    "min_samples_leaf": 24,
    "random_state": 42
}

model_full = HistGradientBoostingRegressor(**hgb_params)

model_compact = Pipeline([
    ("selector", SelectFromModel(
        estimator=ExtraTreesRegressor(
            n_estimators=100,
            min_samples_leaf=2,
            random_state=42,
            n_jobs=-1
        ),
        threshold="0.5*mean"
    )),
    ("model", HistGradientBoostingRegressor(**hgb_params))
])

models = {
    "HistGradientBoosting completo": model_full,
    "HistGradientBoosting compacto": model_compact
}


## 3. Entrenamiento y métricas de test

- **R²:** más alto es mejor.
- **RMSE:** más bajo es mejor; penaliza más los errores grandes.
- **MAE:** más bajo es mejor; error absoluto medio en la unidad del precio.
- **MedAE:** mediana del error absoluto.

In [ ]:
evaluation_rows = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    n_features = (
        int(model.named_steps["selector"].get_support().sum())
        if isinstance(model, Pipeline)
        else X_train.shape[1]
    )

    evaluation_rows.append({
        "Modelo": name,
        "Variables": n_features,
        "R2": r2_score(y_test, y_pred),
        "RMSE": root_mean_squared_error(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "MedAE": median_absolute_error(y_test, y_pred)
    })

evaluation = pd.DataFrame(evaluation_rows).sort_values(
    "R2", ascending=False
).reset_index(drop=True)

display(evaluation.round(4))


## 4. Precios reales frente a predicciones

La línea diagonal representa una predicción perfecta. Cuanto más cerca estén los puntos de la línea, más cercanas son las predicciones a los valores reales.

In [ ]:
for name, y_pred in predictions.items():
    plt.figure(figsize=(7, 6))
    plt.scatter(y_test, y_pred, alpha=0.35)

    low = min(float(y_test.min()), float(np.min(y_pred)))
    high = max(float(y_test.max()), float(np.max(y_pred)))
    plt.plot([low, high], [low, high], linestyle="--")

    plt.xlabel("Precio real")
    plt.ylabel("Precio predicho")
    plt.title(f"Precio real vs. predicho — {name}")
    plt.tight_layout()
    plt.show()


## 5. Análisis de residuos

Residuo = precio real − precio predicho. Un residuo positivo significa que el modelo subestimó el precio; uno negativo, que lo sobreestimó.

In [ ]:
best_name = evaluation.loc[0, "Modelo"]
best_pred = predictions[best_name]

residuals = pd.DataFrame({
    "y_real": np.asarray(y_test),
    "y_predicho": best_pred,
    "residuo": np.asarray(y_test) - best_pred,
    "error_absoluto": np.abs(np.asarray(y_test) - best_pred)
}, index=y_test.index)

display(residuals.head())

plt.figure(figsize=(8, 5))
plt.scatter(best_pred, residuals["residuo"], alpha=0.35)
plt.axhline(0, linestyle="--")
plt.xlabel("Precio predicho")
plt.ylabel("Residuo (real - predicho)")
plt.title(f"Residuos vs. predicciones — {best_name}")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(residuals["residuo"], bins=40)
plt.xlabel("Residuo (real - predicho)")
plt.ylabel("Frecuencia")
plt.title(f"Distribución de residuos — {best_name}")
plt.tight_layout()
plt.show()


## 6. Casos con los errores más grandes

Estos registros pueden revelar alojamientos atípicos, variables omitidas o precios difíciles de predecir. No se deben eliminar automáticamente sin investigar su contexto.

In [ ]:
display(
    residuals.sort_values("error_absoluto", ascending=False).head(15)
)


## 7. Variables conservadas por el modelo compacto

La selección indica utilidad predictiva en estos datos, no causalidad.

In [ ]:
compact_selector = model_compact.named_steps["selector"]
selected_features = X_train.columns[compact_selector.get_support()].tolist()

print(f"Variables seleccionadas: {len(selected_features)} de {X_train.shape[1]}")
for feature in selected_features:
    print(f"- {feature}")


## 8. Conclusiones

Después de ejecutar el notebook, completá las conclusiones con las métricas recién calculadas:

1. ¿Qué modelo obtuvo el mayor R²?
2. ¿Cuál tuvo menor RMSE y MAE?
3. ¿Cuántas variables utiliza el modelo compacto?
4. ¿Qué patrones aparecen en los residuos y en los errores más grandes?
5. ¿Qué limitaciones quedan pendientes?

**Limitación:** el conjunto de test se utilizó durante experimentos previos. Para una evaluación independiente, reservá un test nuevo o aplicá validación cruzada anidada.